In [1]:
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from pronostico_demanda_electrica.baselines import (
    dividir_entrenamiento_prueba,
    naive_estacional,
)
from pronostico_demanda_electrica.features import agregar_variables
from pronostico_demanda_electrica.metrics import evaluar_por_region, wape
from pronostico_demanda_electrica.models import pronosticar_lgbm, pronosticar_ols
from pronostico_demanda_electrica.queries import leer_demanda_mensual
from pronostico_demanda_electrica.storage import crear_engine

load_dotenv()
datos = agregar_variables(leer_demanda_mensual(crear_engine()))
entrenamiento, prueba = dividir_entrenamiento_prueba(datos)
entrenamiento_modelo = entrenamiento.dropna(subset=["log_crecimiento"])
print("Filas para entrenar modelos:", len(entrenamiento_modelo))
print("Filas de prueba:", len(prueba))

Filas para entrenar modelos: 288
Filas de prueba: 108


In [2]:
pronosticos = {
    "naive_estacional": naive_estacional(entrenamiento, prueba),
    "ols": pronosticar_ols(entrenamiento_modelo, prueba),
    "lightgbm": pronosticar_lgbm(entrenamiento_modelo, prueba),
}

comparacion = pd.concat(
    {
        nombre: evaluar_por_region(prueba, p)["wape_%"]
        for nombre, p in pronosticos.items()
    },
    axis=1,
)
comparacion.round(1)

,naive_estacional,ols,lightgbm
region,,,
BCA-BCA,5.7,5.3,6.4
BCS-BCA,10.2,6.1,7.4
SIN-CEN,2.7,2.4,3.7
SIN-NES,3.8,3.4,4.1
SIN-NOR,7.0,5.4,7.1
SIN-NTE,4.4,5.0,5.3
SIN-OCC,2.8,1.5,3.5
SIN-ORI,3.3,2.3,3.2
SIN-PEN,4.3,6.1,4.7


In [3]:
{nombre: round(wape(prueba["demanda_gwh"], p), 2) for nombre, p in pronosticos.items()}

{'naive_estacional': 3.83, 'ols': 3.2, 'lightgbm': 4.28}

In [4]:
crecimiento_ols = (
    np.exp(entrenamiento_modelo.groupby("region")["log_crecimiento"].mean()) - 1
)
(crecimiento_ols * 100).round(1).sort_values(ascending=False)

region
SIN-PEN    6.0
BCS-BCA    5.7
SIN-NOR    3.1
SIN-ORI    2.5
SIN-OCC    1.8
SIN-NES    1.8
SIN-NTE    1.7
BCA-BCA    1.2
SIN-CEN    0.2
Name: log_crecimiento, dtype: float64